**Import Libray CNN**

In [ ]:
import os
import zipfile
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from google.colab import files
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm as tq
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

**Upload File Kaggle.json**

In [ ]:
files.upload()

**Ambil Data Dari Kaggle**

In [ ]:
dataset_id = "arthurwaruwu/DatasetCapstoneFixx"
nama_zip = dataset_id.split("/")[-1] + ".zip"
folder_tujuan = "dataset_sampah"

if not os.path.exists(nama_zip):
    print("File belum ada, mulai download...")
    !kaggle datasets download -d {dataset_id}
else:
    print("File sudah ada, skip download.")

if not os.path.exists(folder_tujuan):
    print("Mulai extract...")
    !unzip -q {nama_zip} -d {folder_tujuan}
else:
    print("Sudah pernah di-extract, skip.")

**Data Preparation**

In [ ]:
# Path sumber data hasil extract (sesuai struktur folder kamu)
base_path = "dataset_sampah/Dataset-Asli"
anorganik_path = os.path.join(base_path, "Anorganik")
organik_path = os.path.join(base_path, "Organik")

# Kumpulkan SEMUA path gambar dari semua subfolder, tapi label-nya cuma 2: Organik / Anorganik
file_name = []
labels = []
full_path = []

# Loop folder Anorganik (semua subfolder di dalamnya dianggap label "Anorganik")
for root, subdirs, files in os.walk(anorganik_path):
    for name in files:
        if name.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path.append(os.path.join(root, name))
            labels.append("Anorganik")
            file_name.append(name)

# Loop folder Organik (semua subfolder di dalamnya dianggap label "Organik")
for root, subdirs, files in os.walk(organik_path):
    for name in files:
        if name.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path.append(os.path.join(root, name))
            labels.append("Organik")
            file_name.append(name)

df = pd.DataFrame({"path": full_path, "file_name": file_name, "labels": labels})

print("Jumlah gambar per kelas (setelah digabung dari semua subkategori):")
print(df.groupby(['labels']).size())
print(f"\nTotal gambar: {len(df)}")

**Data Spliting**

In [ ]:
X = df['path']
y = df['labels']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=300, stratify=y  # stratify biar proporsi kelas seimbang di train & test
)

df_tr = pd.DataFrame({'path': X_train, 'labels': y_train, 'set': 'train'})
df_te = pd.DataFrame({'path': X_test, 'labels': y_test, 'set': 'test'})
df_all = pd.concat([df_tr, df_te], ignore_index=True)

print("\n=====================================================")
print(df_all.groupby(['set', 'labels']).size())
print("=====================================================")

# ============================================
# SALIN FILE KE STRUKTUR FOLDER BARU (Dataset-Final/train|test/Organik|Anorganik)
# ============================================
dataset_path = "Dataset-Final/"

for index, row in tq(df_all.iterrows(), total=len(df_all)):
    file_path = row['path']

    # Buat direktori tujuan kalau belum ada
    target_dir = os.path.join(dataset_path, row['set'], row['labels'])
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    # Tentukan nama file tujuan (pakai nama file asli)
    destination_file_name = os.path.basename(file_path)
    file_dest = os.path.join(target_dir, destination_file_name)

    # Salin file dari sumber ke tujuan (skip kalau sudah ada, hindari duplikat kalau di-run ulang)
    if not os.path.exists(file_dest):
        shutil.copy2(file_path, file_dest)

print("\nProses penyalinan file selesai.")
print(f"Data final tersimpan di folder: {dataset_path}")

**Image Data Generator**

In [ ]:
TRAIN_DIR = "Dataset-Final/train/"
TEST_DIR = "Dataset-Final/test/"

train_organik = os.path.join(TRAIN_DIR, "Organik")
train_anorganik = os.path.join(TRAIN_DIR, "Anorganik")
test_organik = os.path.join(TEST_DIR, "Organik")
test_anorganik = os.path.join(TEST_DIR, "Anorganik")

print("Total gambar Organik di training set:   ", len(os.listdir(train_organik)))
print("Total gambar Anorganik di training set: ", len(os.listdir(train_anorganik)))
print("Total gambar Organik di test set:       ", len(os.listdir(test_organik)))
print("Total gambar Anorganik di test set:     ", len(os.listdir(test_anorganik)))

# Buat objek ImageDataGenerator yang menormalkan gambar (rescale 0-255 jadi 0-1)
datagen = ImageDataGenerator(rescale=1/255., validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1/255.)

train_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    batch_size=32,
    target_size=(128, 128),      # ukuran kecil biar ringan di Colab
    color_mode="rgb",            # RGB karena MobileNet butuh 3 channel warna (beda dari grayscale di project rontgen)
    class_mode='binary',
    subset='training',
    shuffle=True
)

validation_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    batch_size=32,
    target_size=(128, 128),
    color_mode="rgb",
    class_mode='binary',
    subset='validation',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    batch_size=1,
    target_size=(128, 128),
    color_mode="rgb",
    class_mode='binary',
    shuffle=False
)

print("\nMapping kelas (0 dan 1 mengacu ke label apa):")
print(train_generator.class_indices)

**Pembangunan Model**

In [ ]:
# ============================================
# TAHAP 4: PEMBANGUNAN MODEL (TRANSFER LEARNING - MOBILENET)
# ============================================
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Load MobileNet yang sudah dilatih di ImageNet, tanpa bagian klasifikasi akhirnya (include_top=False)
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Bekukan (freeze) semua layer MobileNet, supaya pengetahuan lamanya tidak "terhapus" saat training
base_model.trainable = False

# Susun model: base MobileNet + layer klasifikasi baru di atasnya
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),     # meratakan output MobileNet jadi 1 vektor (lebih ringan dari Flatten biasa)
    Dense(128, activation='relu'),
    Dropout(0.3),                 # mencegah overfitting
    Dense(1, activation='sigmoid')  # output biner: Organik (0/1) vs Anorganik
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# EarlyStopping: training otomatis berhenti kalau validation loss sudah tidak membaik,
# mencegah overfitting dan menghemat waktu training di Colab
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\nMulai training...")
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=validation_generator,
    callbacks=[early_stop]
)

**Visualisasi Grafik**

In [ ]:
# ============================================
# TAHAP 5: VISUALISASI GRAFIK TRAINING
# ============================================
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, 'r', label='Training Accuracy')
plt.plot(epochs_range, val_acc, 'b', label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, 'r', label='Training Loss')
plt.plot(epochs_range, val_loss, 'b', label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

**Evaluasi Model**

In [ ]:
# ============================================
# TAHAP 6: EVALUASI MODEL (CONFUSION MATRIX + CLASSIFICATION REPORT)
# ============================================
from sklearn.metrics import confusion_matrix, classification_report  # <- pastikan ini ada
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
test_generator.reset()

# Prediksi seluruh data test
preds = model.predict(test_generator, verbose=1)
preds_label = (preds > 0.5).astype(int).flatten()

# Ambil label asli dari generator
true_label = test_generator.classes

# Cek mapping kelas (0 = Anorganik atau Organik, tergantung urutan alfabetis folder)
print("Mapping kelas:", test_generator.class_indices)
nama_kelas = list(test_generator.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(true_label, preds_label)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {k}" for k in nama_kelas],
    columns=[f"Predicted {k}" for k in nama_kelas]
)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(true_label, preds_label, target_names=nama_kelas, digits=4))

**Save Model**

In [ ]:
# ============================================
# TAHAP 7: SIMPAN MODEL (aktifkan comment line dibawah ini)
# ============================================
model.save("model_klasifikasi_sampah.h5")
print("Model berhasil disimpan sebagai 'model_klasifikasi_sampah.h5'")

**Upload Gambar Sendiri**

In [ ]:
# ============================================
# TAHAP 8: TES MODEL PAKAI GAMBAR SENDIRI
# ============================================
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

print("Silakan upload foto sampah yang mau kamu tes...")
uploaded = files.upload()

# Ambil nama file yang baru diupload
nama_file = list(uploaded.keys())[0]

# Load gambar, resize sesuai ukuran input model (128x128, sama seperti saat training)
img = image.load_img(nama_file, target_size=(128, 128))
img_array = image.img_to_array(img)
img_array = img_array / 255.0                    # normalisasi, sama seperti saat training (rescale=1/255.)
img_array = np.expand_dims(img_array, axis=0)     # ubah jadi bentuk batch (1, 128, 128, 3)

# Prediksi
hasil_prediksi = model.predict(img_array)
probabilitas = hasil_prediksi[0][0]

# Mapping kelas (ambil otomatis dari generator sebelumnya, 0=Anorganik, 1=Organik)
nama_kelas = list(test_generator.class_indices.keys())
label_prediksi = nama_kelas[1] if probabilitas > 0.5 else nama_kelas[0]
confidence = probabilitas if probabilitas > 0.5 else (1 - probabilitas)

# Tampilkan gambar beserta hasil prediksinya
plt.imshow(img)
plt.axis('off')
plt.title(f"Prediksi: {label_prediksi} ({confidence*100:.2f}% yakin)")
plt.show()

print(f"\nHasil prediksi: {label_prediksi}")
print(f"Tingkat keyakinan: {confidence*100:.2f}%")